# Progressive Growing Generative Adversarial Network (ProGAN) Applied do CelebA Dataset

Source:<P>

https://github.com/rosinality/progressive-gan-pytorch

Adapted:<P>

Antonio Esteves @ UMinho, May 2024<P>

In [ ]:
from   tqdm             import tqdm
import numpy            as     np
from   PIL              import Image
from   math             import sqrt, log2
from   pathlib          import Path
import os
import yaml
import wandb
import time

import torch
from   torch            import nn, optim
from   torch.nn         import init
from   torch.nn         import functional as F
from   torch.autograd   import Variable, grad
from   torch.utils.data import DataLoader, Dataset
from   torchvision      import datasets, transforms, utils
from   torchinfo        import summary
try:
    from natsort import natsorted
except ModuleNotFoundError:
    print(f"[INFO] installing natsort")
    !pip3 install natsort
    from natsort import natsorted

In [ ]:
# Set to False for start the training without loading a saved model
# Set to True for loading a saved model
LOAD_TRAINED_MODEL     = False

# Set to False for training or retraining a model
# Set to True for DO NOT training the model
SKIP_TRAIN_MODEL       = False

# Initial image size for re-training (if not used, set it to 'None')
FORCE_START_IMAGE_SIZE = None

CONFIG_FILE = '../config/progan_celeba_v2_01.yaml'

with open(CONFIG_FILE, 'r') as file:
    try:
        config = yaml.safe_load(file)
    except yaml.YAMLError as exc:
        print(exc)

In [ ]:
print('parameters:')
for key, value in config.items():
    print(f'\t{key}: {value}')

In [ ]:
data_path = Path(config["dataset_path"])
train_dir = data_path / "train"

print(torch.__version__)

# setup device agnostic code

device = "cuda" if torch.cuda.is_available() else "cpu"

print(f'Using {device} for computing')

# Create a directory to store results from this run, if it does not exist

fname=f'results/{config["experiment_name"]}'
os.makedirs(fname, exist_ok=True)

In [ ]:
wandb.login()

In [ ]:
config_wandb = config

wandb.init(
    project = 'OUR_WANDB_PROJECT_ID',
    entity  = 'OUR_WANDB_ENTITY', 
    config  = config_wandb
)

In [ ]:
def time_format(seconds: int) -> str:
    if seconds is not None:
        seconds = int(seconds)
        d = seconds // (3600 * 24)
        h = seconds // 3600 % 24
        m = seconds % 3600 // 60
        s = seconds % 3600 % 60
        if d > 0:
            return '{:02d}D {:02d}H {:02d}m {:02d}s'.format(d, h, m, s)
        elif h > 0:
            return '{:02d}H {:02d}m {:02d}s'.format(h, m, s)
        elif m > 0:
            return '{:02d}m {:02d}s'.format(m, s)
        elif s > 0:
            return '{:02d}s'.format(s)
    return '-'

In [ ]:
def init_linear(linear):
    init.xavier_normal(linear.weight)
    linear.bias.data.zero_()

def init_conv(conv, glu=True):
    init.kaiming_normal(conv.weight)
    if conv.bias is not None:
        conv.bias.data.zero_()

def spectral_norm(module, name='weight'):
    SpectralNorm.apply(module, name)
    return module

In [ ]:
class SpectralNorm:
    def __init__(self, name):
        self.name = name

    def compute_weight(self, module):
        weight     = getattr(module, self.name + '_orig')
        u          = getattr(module, self.name + '_u')
        size       = weight.size()
        weight_mat = weight.contiguous().view(size[0], -1)
        if weight_mat.is_cuda:
            u = u.cuda()
        v         = weight_mat.t() @ u
        v         = v / v.norm()
        u         = weight_mat @ v
        u         = u / u.norm()
        weight_sn = weight_mat / (u.t() @ weight_mat @ v)
        weight_sn = weight_sn.view(*size)

        return weight_sn, Variable(u.data)

    @staticmethod
    def apply(module, name):
        fn = SpectralNorm(name)

        weight = getattr(module, name)
        del module._parameters[name]
        module.register_parameter(name + '_orig', nn.Parameter(weight.data))
        input_size = weight.size(0)
        u = Variable(torch.randn(input_size, 1) * 0.1, requires_grad=False)
        setattr(module, name + '_u', u)
        setattr(module, name, fn.compute_weight(module)[0])

        module.register_forward_pre_hook(fn)

        return fn

    def __call__(self, module, input):
        weight_sn, u = self.compute_weight(module)
        setattr(module, self.name, weight_sn)
        setattr(module, self.name + '_u', u)

In [ ]:
class EqualLR:
    def __init__(self, name):
        self.name = name

    def compute_weight(self, module):
        weight = getattr(module, self.name + '_orig')
        fan_in = weight.data.size(1) * weight.data[0][0].numel()

        return weight * sqrt(2 / fan_in)

    @staticmethod
    def apply(module, name):
        fn = EqualLR(name)

        weight = getattr(module, name)
        del module._parameters[name]
        module.register_parameter(name + '_orig', nn.Parameter(weight.data))
        module.register_forward_pre_hook(fn)

        return fn

    def __call__(self, module, input):
        weight = self.compute_weight(module)
        setattr(module, self.name, weight)


In [ ]:
def equal_lr(module, name='weight'):
    EqualLR.apply(module, name)

    return module

In [ ]:
class PixelNorm(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, input):
        return input / torch.sqrt(torch.mean(input ** 2, dim=1, keepdim=True) + 1e-8)

In [ ]:
class SpectralNormConv2d(nn.Module):
    def __init__(self, *args, **kwargs):
        super().__init__()

        conv = nn.Conv2d(*args, **kwargs)
        init.kaiming_normal(conv.weight)
        conv.bias.data.zero_()
        self.conv = spectral_norm(conv)

    def forward(self, input):
        return self.conv(input)


In [ ]:
class EqualConv2d(nn.Module):
    def __init__(self, *args, **kwargs):
        super().__init__()

        conv = nn.Conv2d(*args, **kwargs)
        conv.weight.data.normal_()
        conv.bias.data.zero_()
        self.conv = equal_lr(conv)

    def forward(self, input):
        return self.conv(input)

In [ ]:
class ConvBlock(nn.Module):
    def __init__(
        self,
        in_channel,
        out_channel,
        kernel_size,
        padding,
        kernel_size2=None,
        padding2=None,
        pixel_norm=True, 
        spectral_norm=False
        ):

        super().__init__()

        pad1 = padding
        pad2 = padding
        if padding2 is not None:
            pad2 = padding2

        kernel1 = kernel_size
        kernel2 = kernel_size
        if kernel_size2 is not None:
            kernel2 = kernel_size2

        if spectral_norm:
            self.conv = nn.Sequential(
                SpectralNormConv2d(
                    in_channel,
                    out_channel, 
                    kernel1,
                    padding=pad1
                ),
                nn.LeakyReLU(0.2),
                SpectralNormConv2d(
                    out_channel,
                    out_channel, 
                    kernel2,
                    padding=pad2
                ),
                nn.LeakyReLU(0.2)
            )

        else:
            if pixel_norm:
                self.conv = nn.Sequential(
                    EqualConv2d(
                        in_channel,
                        out_channel,
                        kernel1, 
                        padding=pad1
                    ),
                    PixelNorm(),
                    nn.LeakyReLU(0.2),
                    EqualConv2d(
                        out_channel,
                        out_channel,
                        kernel2,
                        padding=pad2
                    ),
                    PixelNorm(),
                    nn.LeakyReLU(0.2)
                )

            else:
                self.conv = nn.Sequential(
                    EqualConv2d(
                        in_channel,
                        out_channel,
                        kernel1,
                        padding=pad1
                    ),
                    nn.LeakyReLU(0.2),
                    EqualConv2d(
                        out_channel,
                        out_channel,
                        kernel2, 
                        padding=pad2
                    ),
                    nn.LeakyReLU(0.2)
                )

    def forward(self, input):
        out = self.conv(input)

        return out

In [ ]:
class Generator(nn.Module):

    def __init__(self, code_dim=512 - 10, n_label=10):
        super().__init__()

        self.label_embed = nn.Embedding(n_label, n_label)
        self.code_norm   = PixelNorm()
        self.label_embed.weight.data.normal_()
        self.progression = nn.ModuleList(
            [
                ConvBlock(512, 512, 4, 3, 3, 1),
                ConvBlock(512, 512, 3, 1),
                ConvBlock(512, 512, 3, 1),
                ConvBlock(512, 512, 3, 1),
                ConvBlock(512, 256, 3, 1),
                ConvBlock(256, 128, 3, 1)
            ]
        )

        self.to_rgb = nn.ModuleList(
            [
                nn.Conv2d(512, 3, 1),
                nn.Conv2d(512, 3, 1),
                nn.Conv2d(512, 3, 1),
                nn.Conv2d(512, 3, 1),
                nn.Conv2d(256, 3, 1),
                nn.Conv2d(128, 3, 1)
            ]
        )

    def forward(self, input, label, step=0, alpha=-1):
        input = self.code_norm(input)
        label = self.label_embed(label)

        # print(f'label shape={label.shape} input shape={input.shape}') ################### DEBUG CODE

        out   = torch.cat([input, label], 1).unsqueeze(2).unsqueeze(3)

        for i, (conv, to_rgb) in enumerate(zip(self.progression, self.to_rgb)):
            if i > 0 and step > 0:
                upsample = F.interpolate(out, scale_factor=2)
                out = conv(upsample)

            else:
                out = conv(out)

            if i == step:
                out = to_rgb(out)

                if i > 0 and 0 <= alpha < 1:
                    skip_rgb = self.to_rgb[i - 1](upsample)
                    out = (1 - alpha) * skip_rgb + alpha * out

                break

        return out

In [ ]:
class Discriminator(nn.Module):
    def __init__(self, n_label=10):
        super().__init__()

        self.progression = nn.ModuleList(
            [
                ConvBlock(
                    128,
                    256,
                    3,
                    1,
                    pixel_norm=False,
                    spectral_norm=False
                ),
                ConvBlock(
                    256,
                    512,
                    3,
                    1,
                    pixel_norm=False,
                    spectral_norm=False
                ),
                ConvBlock(
                    512,
                    512, 
                    3, 
                    1,
                    pixel_norm=False,
                    spectral_norm=False
                ),
                ConvBlock(
                    512, 
                    512, 
                    3, 
                    1,
                    pixel_norm=False,
                    spectral_norm=False
                ),
                ConvBlock(
                    512, 
                    512, 
                    3, 
                    1,
                    pixel_norm=False,
                    spectral_norm=False
                ),
                ConvBlock(
                    513, 
                    512, 
                    3, 
                    1, 
                    4, 
                    0,
                    pixel_norm=False,
                    spectral_norm=False
                )
            ]
        )

        self.from_rgb = nn.ModuleList(
            [
                nn.Conv2d(3, 128, 1),
                nn.Conv2d(3, 256, 1),
                nn.Conv2d(3, 512, 1),
                nn.Conv2d(3, 512, 1),
                nn.Conv2d(3, 512, 1),
                nn.Conv2d(3, 512, 1)
            ]
        )

        self.n_layer = len(self.progression)
        self.linear  = nn.Linear(512, 1 + n_label)

    def forward(self, input, step=0, alpha=-1):
        for i in range(step, -1, -1):
            index = self.n_layer - i - 1

            if i == step:
                out = self.from_rgb[index](input)

            if i == 0:
                mean_std = input.std(0).mean()
                mean_std = mean_std.expand(input.size(0), 1, 4, 4)
                out      = torch.cat([out, mean_std], 1)

            out = self.progression[index](out)

            if i > 0:
                out = F.avg_pool2d(out, 2)

                if i == step and 0 <= alpha < 1:
                    skip_rgb = F.avg_pool2d(input, 2)
                    skip_rgb = self.from_rgb[index + 1](skip_rgb)
                    out      = (1 - alpha) * skip_rgb + alpha * out

        out = out.squeeze(2).squeeze(2)
        # print(input.size(), out.size(), step)
        out = self.linear(out)

        return out[:, 0], out[:, 1:]

In [ ]:
def requires_grad(model, flag=True):
    for p in model.parameters():
        p.requires_grad = flag

In [ ]:
def accumulate(model1, model2, decay=0.999):
    part1 = dict(model1.named_parameters())
    part2 = dict(model2.named_parameters())

    for k in part1.keys():
        part1[k].data.mul_(decay).add_(part2[k].data, alpha=1 - decay)

In [ ]:
class CustomDataSet(Dataset):

    def __init__(self, root_dir, transform):
        self.root_dir     = root_dir
        self.transform    = transform
        self.all_images   = os.listdir(root_dir)
        self.total_images = natsorted(self.all_images)

    def __len__(self):
        return len(self.total_images)

    def __getitem__(self, idx):
        img_loc      = os.path.join(self.root_dir, self.total_images[idx])
        image        = Image.open(img_loc).convert("RGB")
        tensor_image = self.transform(image)
        return tensor_image

In [ ]:
def get_loader(image_size, batch_sizes, dataset_train_dir):

    train_transform = transforms.Compose(
        [
            transforms.Resize((image_size, image_size)),
            transforms.ToTensor(),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.Normalize(train_mean, train_stddev),
        ]
    )
    batch_size = batch_sizes[int(log2(image_size / 4))]

    train_data = CustomDataSet(
        root_dir  = dataset_train_dir,
        transform = train_transform,
    )

    train_loader = DataLoader(
        dataset     = train_data,
        batch_size  = batch_size,
        shuffle     = True,
        num_workers = 4,
        pin_memory  = True,
    )

    return train_loader, train_data

In [ ]:
def save_model_and_results(generator, discriminator, generator_accum, results, hyperparameters, file_name):
    results_to_save = {
        'generator':             generator.state_dict(),
        'discriminator':         discriminator.state_dict(),
        'generator_accumulated': generator_accum.state_dict(),
        'results':               results,
        'hyperparameters':       hyperparameters,
    }

    torch.save(
        results_to_save,
        file_name,
    )

In [ ]:
def load_model(generator, discriminator, generator_accum, file_name, device):
    '''
    Given instances of the generator, discriminator and accumulated generator models, 
    loads from file 'file_name':
    (i)   the weights of three models,
    (ii)  the results obtained during model training and
    (iii) the training hyperparameters used to train the models,
    and put the models on 'device'.

    Returns the loaded results and the loaded hyperparameters.
    '''
    results_loaded = torch.load(file_name)

    generator.load_state_dict(results_loaded['generator'])
    generator.to(device)

    discriminator.load_state_dict(results_loaded['discriminator'])
    discriminator.to(device)

    generator_accum.load_state_dict(results_loaded['generator_accumulated'])
    generator_accum.to(device)

    # Returns the saved results and the saved hyperparameters
    return results_loaded['results'], results_loaded['hyperparameters']


In [ ]:
def train_step(
        generator, 
        discriminator,
        generator_accumulated, 
        g_optimizer, 
        d_optimizer,
        n_critic,
        step,
        epoch,
        loader,
        results,
    ):

    requires_grad(generator, False)
    requires_grad(discriminator, True)

    loss_disc = 0
    loss_gen  = 0
    gp        = 0

    alpha         = 0
    one           = torch.tensor(1, dtype=torch.float).to(device)
    mone          = one * -1

    start_time    = time.time()
    pbar          = tqdm(loader, leave=True)

    for i, real_image in enumerate(pbar):

        batch_size = real_image.shape[0]
        alpha      = min(1, 0.00002 * i)

        discriminator.zero_grad()

        label = torch.zeros([batch_size], dtype=torch.int32).to(device)  ######### WE DO NOT HAVE LABEL IN CelebA

        real_image   = Variable(real_image).to(device)
        label        = Variable(label).to(device)
        real_predict, _ = discriminator(real_image, step, alpha)
        real_predict = real_predict.mean() - 0.001 * (real_predict ** 2).mean()
        real_predict.backward(mone)

        fake_image = generator(
            Variable(torch.randn(batch_size, config["code_size"])).to(device),
            label,
            step,
            alpha
        )

        fake_predict, _ = discriminator(
            fake_image,
            step,
            alpha
        )

        fake_predict = fake_predict.mean()
        fake_predict.backward(one)

        eps            = torch.rand(batch_size, 1, 1, 1).to(device)
        x_hat          = eps * real_image.data + (1 - eps) * fake_image.data
        x_hat          = Variable(x_hat, requires_grad=True)
        hat_predict, _ = discriminator(x_hat, step, alpha)
        grad_x_hat     = grad(
            outputs      = hat_predict.sum(),
            inputs       = x_hat,
            create_graph = True
            )[0]
        grad_penalty  = ((grad_x_hat.view(grad_x_hat.size(0), -1)
                         .norm(2, dim=1) - 1)**2).mean()
        grad_penalty  = config["lambda_gp"] * grad_penalty
        gp            = grad_penalty.item()
        grad_penalty.backward()

        loss_disc = (fake_predict - real_predict + grad_penalty).item()
        d_optimizer.step()

        # Save the results in a dictionary
        results["d_loss"].append(loss_disc)
        results["gp"].append(gp)
        results["alpha"].append(alpha)

        if (i + 1) % n_critic == 0:
            generator.zero_grad()

            requires_grad(generator, True)
            requires_grad(discriminator, False)

            input_class = Variable(
                torch.multinomial(
                    torch.ones(config["n_label"]), batch_size, replacement=True
                )
            ).to(device)

            fake_image = generator(
                Variable(torch.randn(batch_size, config["code_size"])).to(device),
                input_class,
                step,
                alpha
            )

            predict, _ = discriminator(fake_image, step, alpha)

            loss     = -predict.mean()
            loss_gen = loss.item()
            loss.backward()
            g_optimizer.step()
            accumulate(generator_accumulated, generator)

            requires_grad(generator, False)
            requires_grad(discriminator, True)

            # Save the generator loss in a dictionary
            results["g_loss"].append(loss_gen)

        # Print progress metrics and save them to W&B
        if i % config["log_interval"] == 0:

            mean_d_loss = np.mean(results["d_loss"][-config["log_interval"]:])
            mean_g_loss = np.mean(results["g_loss"][-config["log_interval"]:])

            try:
                # Log metrics to W&B
                wandb.log(
                    {
                    "discriminator_loss": mean_d_loss,
                    "generator_loss":     mean_g_loss,
                    "alpha":              alpha,
                    "epoch":              epoch+1,
                    "step":               step,
                    }
                )
            except Exception as ex:
                print(f'An exception of type {type(ex).__name__} occurred. Arguments:\n{ex.args!r}')

        pbar.set_postfix(
            step   = step,
            epoch  = epoch,
            GP     = gp,
            D_loss = loss_disc,
            G_loss = loss_gen,
        )

    end_time  = time.time()
    texec_sec = end_time - start_time
    texec_str = time_format(texec_sec)

    results["epoch_training_time"] = texec_sec
    print(f'Epoch training time: {texec_str}')

    try:
        wandb.log(
            {
            "epoch_training_time_sec": texec_sec,
             }
        )
    except Exception as ex:
        print(f'An exception of type {type(ex).__name__} occurred. Arguments:\n{ex.args!r}')

    return alpha

In [ ]:
# code_size = config["z_dim"] - config["n_label"]
config["code_size"] = config["z_dim"] - config["n_label"]

# Create an empty dictionary to store the training results
results = {
    'd_loss': [],
    'g_loss': [],
    'gp':     [],
    'alpha':  [],
    'epoch_training_time': 0.0,
}

train_mean     = torch.tensor([0.5184, 0.4153, 0.3617])
train_variance = torch.tensor([0.0890, 0.0715, 0.0682])
train_stddev   = torch.tensor([0.2983, 0.2674, 0.2611])

print(f'Training data mean:     {train_mean}')
print(f'Training data variance: {train_variance}')
print(f'Training data stddev:   {train_stddev}')

generator             = Generator(config["code_size"], config["n_label"]).to(device)
discriminator         = Discriminator(config["n_label"]).to(device)
generator_accumulated = Generator(config["code_size"], config["n_label"]).to(device)

class_loss  = nn.CrossEntropyLoss()
g_optimizer = optim.Adam(generator.parameters(),     lr=config["lr"], betas=(config["beta1"], config["beta2"]))
d_optimizer = optim.Adam(discriminator.parameters(), lr=config["lr"], betas=(config["beta1"], config["beta2"]))


## Print the model architectures

In [ ]:
step     = int(log2(config["start_image_size"] / 4))
img_size = config["start_image_size"]

for bs in config["batch_sizes"]:
    alpha    = 1e-5
    g_z      = torch.randn(bs, config["code_size"]).to(device)
    g_labels = torch.zeros([bs], dtype=torch.int32).to(device)
    print('###################################################################')
    print(f'Generator architecture in step {step} when batch size is {bs}:')
    print('###################################################################')
    print(summary(generator, input_data=(g_z, g_labels, step, alpha)))
    del g_z
    del g_labels

    d_x = torch.randn(
        bs,
        config["channels_image"],
        img_size,
        img_size,
    ).to(device)
    print('###################################################################')
    print(f'Discriminator architecture  in step {step} when batch size is {bs}:')
    print('###################################################################')
    print(summary(discriminator, input_data=(d_x, step, alpha)))
    del d_x

    img_size *= 2
    step     += 1

In [ ]:
# ==========================================================================================
# Train the model from the beginning
# ==========================================================================================

if LOAD_TRAINED_MODEL == False and SKIP_TRAIN_MODEL == False:

    generator_accumulated.train(False)
    accumulate(generator_accumulated, generator, 0)

    step = int(log2(config["start_image_size"] / 4))

    # ...............................................
    # Loop relative to the progressive growing steps
    # ...............................................

    for num_epochs in config["progressive_epochs"]:

        # 0->4, 1->8, 2->16, 3->32, 4->64, 5->128, 6->256, ...
        loader, dataset = get_loader(4 * 2 ** step, config["batch_sizes"], dataset_train_dir=train_dir)
        print(f"Current image size: {4 * 2 ** step}")

        # .........................................................
        # Loop relative to the epochs of a progressive growing step
        # .........................................................

        for epoch in range(num_epochs):

            print(f"\nEpoch [{epoch+1}/{num_epochs}]")

            # Train the models for one epoch
            alpha = train_step(
                generator,
                discriminator,
                generator_accumulated,
                g_optimizer,
                d_optimizer,
                config["n_critic"],
                step,
                epoch,
                loader,
                results,
            )

            # Save the models to file
            file_save_model = f'models/{config["experiment_name"]}.pth'
            save_model_and_results(
                generator,
                discriminator,
                generator_accumulated,
                results,
                config,
                file_save_model,
            )

            # Generate a grid of images and save them to a file
            images = []
            for _ in range(8):
                input_class = Variable(torch.zeros(8).long()).to(device)
                images.append(
                    generator_accumulated(
                        Variable(torch.randn(config["n_label"] * 8, config["code_size"])).to(device),
                        input_class,
                        step,
                        alpha,
                    ).data.cpu()
                )

            utils.save_image(
                torch.cat(images, 0),
                f'results/{config["experiment_name"]}/{config["experiment_name"]}_generated_step{step}_epoch{str(epoch).zfill(3)}.png',
                nrow        = config["n_label"] * 8,
                normalize   = True,
                value_range = (-1, 1),
            )

        # Progress to the next image size
        step += 1

# ==========================================================================================
# Load the saved models
# ==========================================================================================

elif LOAD_TRAINED_MODEL == True:

    file_save_model = f'models/{config["experiment_name"]}.pth'
    results, _      = load_model(
        generator,
        discriminator,
        generator_accumulated,
        file_save_model,
        device
    )

    # --------------------------------------------------------------------------------------
    # Continue training of the loaded models
    # --------------------------------------------------------------------------------------

    if SKIP_TRAIN_MODEL == False:

        # Put the models in training mode
        generator.train()
        discriminator.train()
        generator_accumulated.train(False)

        if FORCE_START_IMAGE_SIZE is not None:
            start_step = int(log2(FORCE_START_IMAGE_SIZE / 4))
            print(f'Restart training in progressive step {start_step}')
        else:
            print("Initial image size for re-training is not defined, so it is assumed to be 4")
            start_step = 0

        # ...............................................
        # Loop relative to the progressive growing steps
        # ...............................................

        for step in range(start_step, len(config["progressive_epochs"]), 1):

            num_epochs = config["progressive_epochs"][step]

            # 0->4, 1->8, 2->16, 3->32, 4->64, 5->128, 6->256, ...
            loader, dataset = get_loader(4 * 2 ** step, config["batch_sizes"], dataset_train_dir=train_dir)
            print(f"Current image size: {4 * 2 ** step}")

            # .........................................................
            # Loop relative to the epochs of a progressive growing step
            # .........................................................

            for epoch in range(num_epochs):

                print(f"\nEpoch [{epoch+1}/{num_epochs}]")

                # Train the models for one epoch
                alpha = train_step(
                    generator,
                    discriminator,
                    generator_accumulated,
                    g_optimizer,
                    d_optimizer,
                    config["n_critic"],
                    step,
                    epoch,
                    loader,
                    results,
                )

                # Save the models to file
                file_save_model = f'models/{config["experiment_name"]}.pth'
                save_model_and_results(
                    generator,
                    discriminator,
                    generator_accumulated,
                    results,
                    config,
                    file_save_model,
                )

                # Generate a grid of images and save them to a file
                images = []
                for _ in range(8):
                    input_class = Variable(torch.zeros(8).long()).to(device)
                    images.append(
                        generator_accumulated(
                            Variable(torch.randn(config["n_label"] * 8, config["code_size"])).to(device),
                            input_class,
                            step,
                            alpha,
                        ).data.cpu()
                    )

                utils.save_image(
                    torch.cat(images, 0),
                    f'results/{config["experiment_name"]}/{config["experiment_name"]}_generated_step{step}_epoch{str(epoch).zfill(3)}.png',
                    nrow        = config["n_label"] * 8,
                    normalize   = True,
                    value_range = (-1, 1),
                )

            # Progress to the next image size
            step += 1

In [ ]:
def generate_grid_images(g_model, num_grids, grid_W_H, config, device):

    g_model.eval()

    with torch.inference_mode():

        for num in range(num_grids):

            images = []
            for _ in range(grid_W_H):
                input_class = Variable(torch.zeros(grid_W_H).long()).to(device)
                images.append(
                    g_model(
                        input = Variable(torch.randn(config["n_label"] * grid_W_H, config["code_size"])).to(device),
                        label = input_class,
                        step  = 5,
                        alpha = 1.0,
                    ).data.cpu()
                )

            utils.save_image(
                torch.cat(images, 0),
                f'results/{config["experiment_name"]}/{config["experiment_name"]}_generated_final_{str(num+1).zfill(3)}.png',
                nrow        = config["n_label"] * grid_W_H,
                normalize   = True,
                value_range = (-1.0, 1.0),
            )

In [ ]:
generate_grid_images(generator_accumulated, 8, 8, config, device)

In [ ]:
wandb.finish()